# Colab SSH via Ngrok Tunnel

Connect VS Code to a Google Colab GPU runtime via SSH over ngrok.

**Steps:**
1. Install dependencies (openssh-server, pyngrok)
2. Configure SSH server (root login + password auth)
3. Start ngrok TCP tunnel on port 22
4. Connect from VS Code using the printed SSH command


In [1]:
# @title Configuration
NGROK_TOKEN = "2hBGZbcSN2gzulwjHCBG0jCYftM_5uJXfShWQcm4w52sT9Co"  # @param {type:"string"}
SSH_PASSWORD = "axolotl123"  # @param {type:"string"}


In [2]:
# @title 1. Install Dependencies
!pip install pyngrok -q
!apt-get update -qq && apt-get install openssh-server -qq > /dev/null
print("Dependencies installed.")


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Dependencies installed.


In [3]:
# @title 2. Configure SSH Server
import os

# Set root password
os.system(f'echo "root:{SSH_PASSWORD}" | chpasswd')

# Enable root login and password auth
!sed -i 's/#PermitRootLogin prohibit-password/PermitRootLogin yes/' /etc/ssh/sshd_config
!sed -i 's/PermitRootLogin without-password/PermitRootLogin yes/' /etc/ssh/sshd_config
!sed -i 's/#PasswordAuthentication yes/PasswordAuthentication yes/' /etc/ssh/sshd_config

# Restart SSH service
!service ssh restart
print("SSH server configured and started.")


 * Restarting OpenBSD Secure Shell server sshd
   ...done.
SSH server configured and started.


In [4]:
# @title 3. Start Ngrok Tunnel
from pyngrok import ngrok, conf

# Kill any existing ngrok processes
!pkill ngrok 2>/dev/null || true

# Configure and authenticate ngrok
conf.get_default().auth_token = NGROK_TOKEN

# Open TCP tunnel to SSH port
ssh_tunnel = ngrok.connect(22, "tcp")
print(f"Ngrok tunnel established: {ssh_tunnel.public_url}")


Ngrok tunnel established: tcp://6.tcp.ngrok.io:19024                                                


In [5]:
# @title 4. SSH Connection Details
from urllib.parse import urlparse

parsed = urlparse(ssh_tunnel.public_url)
hostname = parsed.hostname
port = parsed.port

print("=" * 60)
print("SSH CONNECTION DETAILS")
print("=" * 60)
print(f"  Host:     {hostname}")
print(f"  Port:     {port}")
print(f"  User:     root")
print(f"  Password: {SSH_PASSWORD}")
print()
print(f"  Command:  ssh root@{hostname} -p {port}")
print("=" * 60)
print()
print("VS Code Remote SSH config (~/.ssh/config):")
print(f"""
Host colab
    HostName {hostname}
    Port {port}
    User root
""")


SSH CONNECTION DETAILS
  Host:     6.tcp.ngrok.io
  Port:     19024
  User:     root
  Password: axolotl123

  Command:  ssh root@6.tcp.ngrok.io -p 19024

VS Code Remote SSH config (~/.ssh/config):

Host colab
    HostName 6.tcp.ngrok.io
    Port 19024
    User root



## Keep Alive

Run the cell below to keep the Colab runtime active while you work in VS Code.
The tunnel will stay open as long as this cell is running.


In [6]:
# @title 5. Keep Alive (run this to prevent Colab timeout)
import time
print("Keeping Colab alive... Press stop to disconnect.")
print(f"Tunnel: {ssh_tunnel.public_url}")
while True:
    time.sleep(60)
    print(".", end="", flush=True)


Keeping Colab alive... Press stop to disconnect.
Tunnel: tcp://6.tcp.ngrok.io:19024
..........................................................................................................................................................................

: 

## Alternative: Cloudflare Tunnel (no ngrok account needed)

If ngrok doesn't work or you prefer Cloudflare:


In [ ]:
# @title Alternative: Cloudflare Tunnel
# Uncomment and run if you prefer cloudflared over ngrok

# !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
# !chmod +x /usr/local/bin/cloudflared
# !cloudflared tunnel --url ssh://localhost:22
